In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Fetch the data

In [0]:
import tomllib
import sys
from datetime import datetime, timezone

user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")

config_path = f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/config.toml"
with open(config_path, "rb") as f:
    config = tomllib.load(f)

from_dt = datetime.combine(config["dates"]["historical"]["from_date"], datetime.min.time()).replace(tzinfo=timezone.utc)
to_dt   = datetime.combine(config["dates"]["historical"]["to_date"], datetime.min.time()).replace(tzinfo=timezone.utc)

from src.data.collectors.carbon_intensity import fetch_regional_carbon_intensity_by_region_in_chunks


### Fetch Data in Chunks

Fetching ~1 year of data in 7 day chunks

In [0]:
df_bronze = fetch_regional_carbon_intensity_by_region_in_chunks(from_dt, to_dt)
display(df_bronze.head(50))

### Write a Delta Table for the Bronze layer

In [0]:
from pyspark.sql.functions import col

sdf = spark.createDataFrame(df_bronze)
sdf = sdf.filter(col("region_id") <= 14)

sdf.createOrReplaceTempView("carbon_intensity_staging")
spark.sql("CREATE TABLE IF NOT EXISTS bronze_carbon_intensity_regional USING DELTA AS SELECT * FROM carbon_intensity_staging WHERE 1=0")

spark.sql("""
MERGE INTO bronze_carbon_intensity_regional
  USING carbon_intensity_staging
  ON bronze_carbon_intensity_regional.region_id = carbon_intensity_staging.region_id
  AND bronze_carbon_intensity_regional.period_from = carbon_intensity_staging.period_from
  WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
# I want to get the latest and earliest 5 times from period_from
earliest_5 = df_bronze.sort_values("period_from").head(5)
latest_5 = df_bronze.sort_values("period_from", ascending=False).head(5)


In [ ]:
# Sanity check — verify date range of fetched data
earliest = df_bronze.sort_values("period_from").iloc[0]["period_from"]
latest = df_bronze.sort_values("period_from", ascending=False).iloc[0]["period_from"]
print(f"Date range: {earliest} → {latest}")
print(f"Rows fetched (pandas, pre-filter): {len(df_bronze)}")
print(f"Rows written to Delta (post region_id filter): {spark.table('bronze_carbon_intensity_regional').count()}")

In [0]:
print(f"Rows: {len(df_bronze)}")
print(f"Columns: {len(df_bronze.columns)}") 